# Agent 1 — Module 3 Final Testing Notebook

This notebook tests the **six current production `.docx` transcripts**:

1. `Transcript 1.docx`
2. `Transcript_Raw_2_Networks_Cybersecurity.docx`
3. `Transcript_Raw_3_Algorithms_Programming.docx`
4. `Transcript_Test_1_Data_Representation.docx`
5. `Transcript_Test_2_Networks_Cybersecurity.docx`
6. `Transcript_Test_3_Algorithms_Programming_Stress.docx`

Every transcript is processed through the current production Agent 1 pipeline.

Outputs are stored inside:

```text
tester/
└── <transcript_name>/
    ├── 01_preprocessing/
    ├── 02_chunking/
    └── 03_topic_extraction/
```

This keeps the cleaned transcript, chunking files, and complete topic-generation files together for every transcript.

## 1. Project setup

Keep this notebook in the `Agent_1` root folder:

```text
Agent_1/
├── app/
├── scripts/
├── test_data/
├── requirements.txt
└── Agent1_Module3_Final_Tester_Updated.ipynb
```

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

required_items = [
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "scripts",
    PROJECT_ROOT / "test_data",
]

missing_items = [
    path.name
    for path in required_items
    if not path.exists()
]

if missing_items:
    raise RuntimeError(
        "This notebook must be run from the Agent_1 root folder. "
        f"Missing: {', '.join(missing_items)}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.executable}")
print(f"Working dir  : {Path.cwd()}")

## 2. Verify the project virtual environment

The active Python should normally point to:

```text
Agent_1/.venv/Scripts/python.exe
```

In [ ]:
expected_venv = (PROJECT_ROOT / ".venv").resolve()
active_python = Path(sys.executable).resolve()

using_project_venv = (
    expected_venv in active_python.parents
)

print(f"Expected venv       : {expected_venv}")
print(f"Active Python       : {active_python}")
print(f"Using project .venv : {using_project_venv}")

if not using_project_venv:
    print(
        "\nWARNING: Select the Agent_1 .venv kernel "
        "before running the complete notebook."
    )

## 3. Command runner

The notebook calls your existing production scripts instead of copying Module 1, Module 2, or Module 3 code.

In [ ]:
def run_module(
    module: str,
    arguments: Iterable[str] = (),
    *,
    stop_on_error: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        "-m",
        module,
        *list(arguments),
    ]

    print("\n" + "=" * 110)
    print("RUNNING:")
    print(
        " ".join(
            f'"{part}"' if " " in part else part
            for part in command
        )
    )
    print("=" * 110 + "\n")

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_lines: list[str] = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    result = subprocess.CompletedProcess(
        args=command,
        returncode=return_code,
        stdout="".join(output_lines),
        stderr=None,
    )

    print("\n" + "-" * 110)
    print(f"Exit code: {return_code}")
    print("-" * 110)

    if stop_on_error and return_code != 0:
        raise RuntimeError(
            f"Command failed with exit code {return_code}: "
            f"python -m {module}"
        )

    return result

# Part A — Run all regression tests

These tests verify that the current Module 3 improvements are still working before transcript processing begins.

In [ ]:
module_3_regression_result = run_module(
    "scripts.test_module_3_regressions"
)

In [ ]:
module_1_2_regression_result = run_module(
    "scripts.test_module_1_2_regressions"
)

# Part B — Select the six production transcripts

Only the six current `.docx` transcript files are selected.

PDF strategy/noisy transcripts, cleaned `.txt` outputs, chunk files, and previous generated reports are not included.

In [ ]:
TEST_DATA_DIR = PROJECT_ROOT / "test_data"

TRANSCRIPTS_TO_TEST = [
    TEST_DATA_DIR / "Transcript 1.docx",
    TEST_DATA_DIR / "Transcript_Raw_2_Networks_Cybersecurity.docx",
    TEST_DATA_DIR / "Transcript_Raw_3_Algorithms_Programming.docx",
    TEST_DATA_DIR / "Transcript_Test_1_Data_Representation.docx",
    TEST_DATA_DIR / "Transcript_Test_2_Networks_Cybersecurity.docx",
    TEST_DATA_DIR / "Transcript_Test_3_Algorithms_Programming_Stress.docx",
]

missing_files = [
    path
    for path in TRANSCRIPTS_TO_TEST
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(
        f"- {path.relative_to(PROJECT_ROOT)}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "The following required transcript files were not found:\n"
        f"{missing_text}"
    )

print(
    f"Total transcripts selected: "
    f"{len(TRANSCRIPTS_TO_TEST)}"
)
print()

for index, path in enumerate(
    TRANSCRIPTS_TO_TEST,
    start=1,
):
    print(
        f"{index:>2}. "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

## Transcript selection

The six production transcripts are defined explicitly in the cell above.

To add another production transcript later, add its `.docx` path to `TRANSCRIPTS_TO_TEST`.

In [ ]:
# Example for adding another transcript later:
#
# TRANSCRIPTS_TO_TEST.append(
#     TEST_DATA_DIR / "Transcript_Test_4_New_Topic.docx"
# )
#
# Always verify that the file exists:
#
# for path in TRANSCRIPTS_TO_TEST:
#     if not path.exists():
#         raise FileNotFoundError(path)

# Part C — Prepare the `tester` output folder

By default, the notebook deletes the previous `tester` folder before a fresh run.

This avoids mixing old and new outputs.

In [ ]:
TESTER_OUTPUT_ROOT = PROJECT_ROOT / "tester"

RESET_TESTER_FOLDER = True

if RESET_TESTER_FOLDER and TESTER_OUTPUT_ROOT.exists():
    shutil.rmtree(TESTER_OUTPUT_ROOT)

TESTER_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Tester output folder: {TESTER_OUTPUT_ROOT}")

# Part D — Run Transcript 1 and every batch transcript

Each file is processed using:

```powershell
python -m scripts.run_agent1_pipeline --file "<path>" --output-root "tester" --run-name "<transcript_name>" --no-llm
```

The production pipeline creates the preprocessing, chunking, and topic-generation outputs.

In [ ]:
def safe_run_name(path: Path) -> str:
    name = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        path.stem,
    )
    return name.strip("._-") or "transcript"


run_records: list[dict[str, object]] = []

for index, transcript_path in enumerate(
    TRANSCRIPTS_TO_TEST,
    start=1,
):
    run_name = safe_run_name(
        transcript_path
    )

    print(
        f"\nPROCESSING {index}/{len(TRANSCRIPTS_TO_TEST)}: "
        f"{transcript_path.name}"
    )

    result = run_module(
        "scripts.run_agent1_pipeline",
        [
            "--file",
            str(transcript_path),
            "--output-root",
            str(TESTER_OUTPUT_ROOT),
            "--run-name",
            run_name,
            "--no-llm",
        ],
        stop_on_error=False,
    )

    run_records.append(
        {
            "number": index,
            "transcript": transcript_path.name,
            "source_path": str(
                transcript_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "run_name": run_name,
            "status": (
                "PASS"
                if result.returncode == 0
                else "FAIL"
            ),
            "exit_code": result.returncode,
            "output_folder": str(
                (
                    TESTER_OUTPUT_ROOT
                    / run_name
                ).relative_to(
                    PROJECT_ROOT
                )
            ),
        }
    )

run_summary_df = pd.DataFrame(
    run_records
)

display(run_summary_df)

## Stop if any transcript failed

In [ ]:
failed_runs = run_summary_df[
    run_summary_df["status"] != "PASS"
]

if not failed_runs.empty:
    display(failed_runs)
    raise RuntimeError(
        f"{len(failed_runs)} transcript run(s) failed. "
        "Review the command output above."
    )

print(
    f"ALL {len(run_summary_df)} TRANSCRIPTS "
    "PROCESSED SUCCESSFULLY"
)

# Part E — Verify the required output files

For every transcript, the notebook checks that the following production stage folders exist:

```text
01_preprocessing
02_chunking
03_topic_extraction
```

In [ ]:
REQUIRED_STAGE_FOLDERS = [
    "01_preprocessing",
    "02_chunking",
    "03_topic_extraction",
]

output_checks: list[dict[str, object]] = []

for record in run_records:
    run_folder = (
        TESTER_OUTPUT_ROOT
        / str(record["run_name"])
    )

    stage_status = {
        stage: (
            run_folder / stage
        ).is_dir()
        for stage in REQUIRED_STAGE_FOLDERS
    }

    files_by_stage = {
        stage: len(
            [
                path
                for path in (
                    run_folder / stage
                ).rglob("*")
                if path.is_file()
            ]
        )
        if (
            run_folder / stage
        ).is_dir()
        else 0
        for stage in REQUIRED_STAGE_FOLDERS
    }

    output_checks.append(
        {
            "transcript": record["transcript"],
            "run_folder": str(
                run_folder.relative_to(
                    PROJECT_ROOT
                )
            ),
            "preprocessing_folder": stage_status[
                "01_preprocessing"
            ],
            "preprocessing_files": files_by_stage[
                "01_preprocessing"
            ],
            "chunking_folder": stage_status[
                "02_chunking"
            ],
            "chunking_files": files_by_stage[
                "02_chunking"
            ],
            "topic_extraction_folder": stage_status[
                "03_topic_extraction"
            ],
            "topic_extraction_files": files_by_stage[
                "03_topic_extraction"
            ],
        }
    )

output_check_df = pd.DataFrame(
    output_checks
)

display(output_check_df)

missing_stage_outputs = output_check_df[
    (
        ~output_check_df["preprocessing_folder"]
        | ~output_check_df["chunking_folder"]
        | ~output_check_df["topic_extraction_folder"]
    )
]

if not missing_stage_outputs.empty:
    display(missing_stage_outputs)
    raise RuntimeError(
        "One or more transcript runs are missing "
        "required pipeline output folders."
    )

print(
    "All required preprocessing, chunking, and "
    "topic-extraction folders were created."
)

# Part F — Save a tester summary

A CSV and JSON summary are saved directly inside the `tester` folder.

In [ ]:
TESTER_SUMMARY_CSV = (
    TESTER_OUTPUT_ROOT
    / "tester_summary.csv"
)

TESTER_SUMMARY_JSON = (
    TESTER_OUTPUT_ROOT
    / "tester_summary.json"
)

run_summary_df.to_csv(
    TESTER_SUMMARY_CSV,
    index=False,
)

with TESTER_SUMMARY_JSON.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_records,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved: {TESTER_SUMMARY_CSV}")
print(f"Saved: {TESTER_SUMMARY_JSON}")

# Part G — Display outputs for a selected transcript

`SELECTED_TRANSCRIPT_INDEX = 0` displays Transcript 1 because Transcript 1 is always placed first.

In [ ]:
SELECTED_TRANSCRIPT_INDEX = 0

if not (
    0
    <= SELECTED_TRANSCRIPT_INDEX
    < len(run_records)
):
    raise IndexError(
        "SELECTED_TRANSCRIPT_INDEX is outside "
        "the available transcript range."
    )

selected_record = run_records[
    SELECTED_TRANSCRIPT_INDEX
]

selected_folder = (
    TESTER_OUTPUT_ROOT
    / str(selected_record["run_name"])
)

print(
    "Selected transcript:",
    selected_record["transcript"],
)
print(
    "Output folder:",
    selected_folder.relative_to(
        PROJECT_ROOT
    ),
)

## Display cleaned transcript

In [ ]:
cleaned_candidates = [
    selected_folder
    / "01_preprocessing"
    / "cleaned_transcript.txt",
    selected_folder
    / "01_preprocessing"
    / "deterministic_cleaned.txt",
]

cleaned_file = next(
    (
        path
        for path in cleaned_candidates
        if path.exists()
    ),
    None,
)

if cleaned_file is None:
    print(
        "No cleaned transcript text file was found."
    )
else:
    print(
        cleaned_file.relative_to(
            PROJECT_ROOT
        )
    )
    print("=" * 110)
    print(
        cleaned_file.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

## Display readable chunks

In [ ]:
chunk_candidates = [
    selected_folder
    / "02_chunking"
    / "chunks_readable.txt",
    selected_folder
    / "02_chunking"
    / "chunks.txt",
]

chunk_file = next(
    (
        path
        for path in chunk_candidates
        if path.exists()
    ),
    None,
)

if chunk_file is None:
    print(
        "No readable chunking file was found."
    )
else:
    print(
        chunk_file.relative_to(
            PROJECT_ROOT
        )
    )
    print("=" * 110)
    print(
        chunk_file.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

## Display complete readable topic-generation report

This cell displays the production readable report from `03_topic_extraction`.

It should contain:

```text
PRIMARY TOPICS
SUPPORTING TOPICS
UNMAPPED / EXTENDED TOPICS
DETAILED MERGED OFFICIAL TOPICS
```

In [ ]:
topic_folder = (
    selected_folder
    / "03_topic_extraction"
)

if not topic_folder.exists():
    raise FileNotFoundError(
        "Topic extraction folder was not found: "
        f"{topic_folder}"
    )

preferred_topic_files = [
    topic_folder / "topics_readable.txt",
    topic_folder / "module_3_readable.txt",
    topic_folder / "topic_extraction_readable.txt",
    topic_folder / "merged_topics_readable.txt",
]

existing_preferred_files = [
    path
    for path in preferred_topic_files
    if path.exists()
]

all_text_files = sorted(
    path
    for path in topic_folder.glob("*.txt")
    if path.is_file()
)

candidate_files = (
    existing_preferred_files
    + [
        path
        for path in all_text_files
        if path not in existing_preferred_files
    ]
)

def readable_report_score(path: Path) -> tuple[int, int]:
    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    required_headings = [
        "PRIMARY TOPICS",
        "SUPPORTING TOPICS",
        "UNMAPPED / EXTENDED TOPICS",
        "DETAILED MERGED OFFICIAL TOPICS",
    ]

    heading_score = sum(
        heading in text
        for heading in required_headings
    )

    return (
        heading_score,
        len(text),
    )

topic_file = (
    max(
        candidate_files,
        key=readable_report_score,
    )
    if candidate_files
    else None
)

if topic_file is None:
    print(
        "No readable topic-generation text file "
        "was found inside:"
    )
    print(topic_folder)

    available_files = sorted(
        path.name
        for path in topic_folder.iterdir()
    )

    if available_files:
        print("\nAvailable files:")
        for filename in available_files:
            print(f"- {filename}")

else:
    topic_output = topic_file.read_text(
        encoding="utf-8",
        errors="replace",
    )

    print(
        "Displaying:",
        topic_file.relative_to(
            PROJECT_ROOT
        ),
    )
    print("=" * 110)
    print(topic_output)

    expected_headings = [
        "PRIMARY TOPICS",
        "SUPPORTING TOPICS",
    ]

    missing_headings = [
        heading
        for heading in expected_headings
        if heading not in topic_output
    ]

    if missing_headings:
        print(
            "\nWARNING: The selected readable report "
            "does not contain these expected headings:"
        )

        for heading in missing_headings:
            print(f"- {heading}")

        print(
            "\nAvailable topic extraction files:"
        )

        for path in sorted(
            topic_folder.iterdir()
        ):
            if path.is_file():
                print(f"- {path.name}")

# Final tester-folder structure

After a successful run, the project will contain:

```text
Agent_1/
└── tester/
    ├── tester_summary.csv
    ├── tester_summary.json
    ├── Transcript_1/
    │   ├── 01_preprocessing/
    │   ├── 02_chunking/
    │   └── 03_topic_extraction/
    ├── Transcript_Raw_2_Networks_Cybersecurity/
    │   ├── 01_preprocessing/
    │   ├── 02_chunking/
    │   └── 03_topic_extraction/
    ├── Transcript_Raw_3_Algorithms_Programming/
    │   ├── 01_preprocessing/
    │   ├── 02_chunking/
    │   └── 03_topic_extraction/
    ├── Transcript_Test_1_Data_Representation/
    ├── Transcript_Test_2_Networks_Cybersecurity/
    └── Transcript_Test_3_Algorithms_Programming_Stress/
```

The topic display cell selects the most complete readable production report and prioritises the file containing grouped **Primary**, **Supporting**, **Unmapped**, and detailed merged-topic sections.